# Data cleaning & visualization — Netflix / IMDb title catalog

**Project:** Data-Cleaning-Project  
**Primary data:** `n_movies.csv` from your **`Downloads\netfilx`** folder (IMDb-style fields: title, year, certificate, duration, genre, rating, votes, …).

### What this notebook does

| Step | Action |
|------|--------|
| 1 | Resolve paths and import shared cleaning code from `src/cleaning.py` |
| 2 | Load CSV and derive numeric columns (`start_year`, `votes_numeric`, `duration_minutes`) |
| 3 | Explore shape, dtypes, and missing values |
| 4 | Fill missing values (text → `Unknown`, numbers → median) |
| 5 | Drop duplicates (`title` for `n_movies`, `show_id` for the tiny demo) |
| 6 | Remove year outliers with the IQR rule |
| 7 | Save **`data/cleaned_n_movies.csv`** (or `cleaned_netflix.csv` for the demo) |
| 8 | Build four charts and save PNGs under **`visuals/`** |

**Run order:** run all cells from top to bottom (**Run All**).  
**Fallback order:** `data/n_movies.csv` → `data/netfilx/n_movies.csv` → `data/raw_data.csv`. If none exist, set `DATA_PATH_OVERRIDE` or you will get a clear error.

## 1. Setup — imports, theme, and paths

- **`PROJECT_ROOT`** is detected whether you launch Jupyter from the repo root or from `notebooks/`.
- **`DATA_PATH_OVERRIDE`**: set to a `Path(...)` if your CSV lives somewhere other than the defaults.
- Outputs always go to **`visuals/`** and **`data/`** inside the project (your original CSV is never overwritten).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Consistent plot style (readable labels, good defaults)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

def _project_has_data_dir(root: Path) -> bool:
    d = root / "data"
    return (
        (d / "n_movies.csv").exists()
        or (d / "raw_data.csv").exists()
        or (d / "netfilx" / "n_movies.csv").exists()
    )


_here = Path.cwd().resolve()
if _project_has_data_dir(_here):
    PROJECT_ROOT = _here
elif _project_has_data_dir(_here.parent):
    PROJECT_ROOT = _here.parent
else:
    PROJECT_ROOT = Path("..").resolve()

# Priority: override → Downloads\netfilx → data/n_movies.csv → data/netfilx/n_movies.csv → raw_data demo
DATA_PATH_OVERRIDE = None  # e.g. Path(r"D:\datasets\n_movies.csv")
DATASET_FROM_DOWNLOADS = Path.home() / "Downloads" / "netfilx" / "n_movies.csv"
_data_n = PROJECT_ROOT / "data" / "n_movies.csv"
_data_sub = PROJECT_ROOT / "data" / "netfilx" / "n_movies.csv"
_data_raw = PROJECT_ROOT / "data" / "raw_data.csv"

if DATA_PATH_OVERRIDE is not None and Path(DATA_PATH_OVERRIDE).is_file():
    DATA_PATH = Path(DATA_PATH_OVERRIDE).resolve()
elif DATASET_FROM_DOWNLOADS.is_file():
    DATA_PATH = DATASET_FROM_DOWNLOADS
elif _data_n.is_file():
    DATA_PATH = _data_n
elif _data_sub.is_file():
    DATA_PATH = _data_sub
elif _data_raw.is_file():
    DATA_PATH = _data_raw
else:
    raise FileNotFoundError(
        "No dataset found. Add Downloads\\netfilx\\n_movies.csv, or data\\n_movies.csv, "
        "or data\\netfilx\\n_movies.csv, or data\\raw_data.csv — or set DATA_PATH_OVERRIDE."
    )

IS_N_MOVIES = DATA_PATH.name == "n_movies.csv"
CLEAN_PATH = (
    PROJECT_ROOT / "data" / "cleaned_n_movies.csv"
    if IS_N_MOVIES
    else PROJECT_ROOT / "data" / "cleaned_netflix.csv"
)
VISUALS_DIR = PROJECT_ROOT / "visuals"
VISUALS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / "src"))
import cleaning as cl  # noqa: E402

print("Project root:", PROJECT_ROOT)
print("Dataset file:", DATA_PATH)
print("Using n_movies schema:", IS_N_MOVIES)

## 2. Data dictionary (`n_movies.csv`)

| Column | Meaning |
|--------|---------|
| `title` | Show or movie title |
| `year` | IMDb-style range string, e.g. `(2016– )` or `(2008–2013)` |
| `certificate` | Content rating (TV-MA, PG-13, …); may be empty |
| `duration` | e.g. `45 min` (episode or feature length) |
| `genre` | Comma-separated genres |
| `rating` | IMDb user rating (numeric) |
| `description` | Short synopsis |
| `stars` | Cast / credit string |
| `votes` | Vote count, often with commas |

**Derived in code:** `start_year` (first year in `year`), `votes_numeric`, `duration_minutes`.

## 3. Load the dataset

In [ ]:
df = cl.load_csv(DATA_PATH)
raw_rows = len(df)

if IS_N_MOVIES:
    df = cl.enrich_n_movies_schema(df)

df.head(10)

## 4. Explore — shape, types, missing values

Always inspect before cleaning: row count, column types, and where data is missing.

In [ ]:
cl.dataset_info(df)

## 5. Handle missing values

Strategy (see `src/cleaning.py`):

- **Object / text columns** → fill with `"Unknown"` so plots and groupbys do not drop rows silently.
- **Numeric columns** → fill with the **median** (robust to skewed distributions).

In [ ]:
df_clean = cl.handle_missing_values(df, fill_text="Unknown")
print("Missing values after fill (should be 0 for used columns):")
print(df_clean.isna().sum())

## 6. Remove duplicate rows

- **`n_movies.csv`:** duplicates are dropped by **`title`** (first row kept). *Note:* different titles can still refer to remakes; this is a simple default.
- **`raw_data.csv`:** duplicates dropped by **`show_id`**, then any fully identical rows.

In [ ]:
before = len(df_clean)
if IS_N_MOVIES:
    df_clean = cl.remove_duplicates(df_clean, subset=["title"])
else:
    df_clean = cl.remove_duplicates(df_clean, subset=["show_id"])
df_clean = cl.remove_duplicates(df_clean)
rows_after_dedupe = len(df_clean)
print(f"Rows before: {before}, after deduplication: {rows_after_dedupe}")

## 7. Outliers — IQR on release / start year

Rows with **start_year** (or **release_year** in the demo) outside \(Q1 - 1.5 \times IQR\) to \(Q3 + 1.5 \times IQR\) are removed. Increase `factor` toward `3.0` for a gentler filter.

In [ ]:
year_col = "start_year" if IS_N_MOVIES else "release_year"
before = len(df_clean)
df_clean = cl.remove_outliers_iqr(df_clean, column=year_col, factor=1.5)
rows_final = len(df_clean)
print(f"Rows before outlier removal: {before}, after: {rows_final}")
df_clean[year_col].describe()

## 8. Save cleaned dataset

Cleaned data is written under **`data/`** in the project. Your source file in **Downloads** is read-only here.

In [ ]:
cl.save_cleaned_csv(df_clean, CLEAN_PATH)
print("Saved:", CLEAN_PATH)

## 9. Prepare visualization table

For **`raw_data.csv`**, `duration_minutes` is parsed here. For **`n_movies`**, it already exists; we only impute any remaining missing minutes with the median for plotting.

In [ ]:
df_viz = df_clean.copy()
if not IS_N_MOVIES:
    df_viz["duration_minutes"] = cl.parse_duration_minutes(df_viz["duration"])

med_dur = df_viz["duration_minutes"].median()
df_viz["duration_minutes"] = df_viz["duration_minutes"].fillna(med_dur)
df_viz.head()

## 10. Histogram — year distribution

Shows how many titles fall into each **start / release year** after cleaning.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df_viz, x=year_col, bins=30, kde=True, color="#e50914")
plt.title("Catalog titles by start year (after cleaning)")
plt.xlabel("Start year")
plt.ylabel("Number of titles")
plt.tight_layout()
hist_path = VISUALS_DIR / "histogram_release_year.png"
plt.savefig(hist_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", hist_path)

## 11. Count plot — certificates or content type

For **`n_movies`**: horizontal bar counts for the **10 most common** age/content certificates.  
For **`raw_data`**: Movies vs TV Shows.

In [ ]:
plt.figure(figsize=(10, 6))
if IS_N_MOVIES:
    top_cert = df_viz["certificate"].value_counts().nlargest(10).index
    sub = df_viz[df_viz["certificate"].isin(top_cert)]
    sns.countplot(data=sub, y="certificate", order=top_cert, palette="rocket")
    plt.title("Top 10 content certificates in catalog")
    plt.xlabel("Title count")
    plt.ylabel("Certificate")
else:
    order = df_viz["type"].value_counts().index
    sns.countplot(data=df_viz, x="type", order=order, palette="rocket")
    plt.title("Titles by content type")
    plt.xlabel("Type")
    plt.ylabel("Count")
plt.tight_layout()
count_path = VISUALS_DIR / "countplot_content_type.png"
plt.savefig(count_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", count_path)

## 12. Scatter — year vs duration

Each point is one title. Large catalogs use smaller, semi-transparent points.

In [ ]:
plt.figure(figsize=(9, 5))
hue_col = "type" if not IS_N_MOVIES else None
sns.scatterplot(
    data=df_viz,
    x=year_col,
    y="duration_minutes",
    hue=hue_col,
    s=14 if IS_N_MOVIES else 80,
    alpha=0.35 if IS_N_MOVIES else 0.85,
    edgecolor=None,
)
plt.title("Start year vs runtime (minutes)")
plt.xlabel("Start year")
plt.ylabel("Duration (minutes)")
if hue_col:
    plt.legend(title="Type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
scatter_path = VISUALS_DIR / "scatter_year_vs_duration.png"
plt.savefig(scatter_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", scatter_path)

## 13. Correlation heatmap

Only **numeric** columns are included (e.g. `start_year`, `duration_minutes`, IMDb `rating`, `votes_numeric`). Text ratings in the demo file are excluded automatically.

In [ ]:
_cands = [year_col, "duration_minutes", "rating", "votes_numeric"]
heatmap_cols = [
    c
    for c in _cands
    if c in df_viz.columns and pd.api.types.is_numeric_dtype(df_viz[c])
]
corr = df_viz[heatmap_cols].corr()

plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, cmap="RdBu_r", center=0, fmt=".2f", linewidths=0.5)
plt.title("Correlation between numeric features")
plt.tight_layout()
heat_path = VISUALS_DIR / "correlation_heatmap.png"
plt.savefig(heat_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", heat_path)

## 14. Summary for `reports/final_report.md`

Run the cell below and paste the printed block into your report (edit the narrative sections yourself).

In [ ]:
_mean_rating = (
    df_viz["rating"].mean()
    if "rating" in df_viz.columns and pd.api.types.is_numeric_dtype(df_viz["rating"])
    else float("nan")
)
_rating_line = (
    f"Mean IMDb rating: {_mean_rating:.2f}"
    if not np.isnan(_mean_rating)
    else "Mean IMDb rating: N/A (non-numeric in demo mode)"
)
summary = f"""
--- Copy into final_report.md ---
Source file (read): {DATA_PATH}
Cleaned output: {CLEAN_PATH}
Raw rows loaded: {raw_rows}
Rows after deduplication: {rows_after_dedupe}
Rows after outlier removal: {rows_final}
Year column used: {year_col}
Year min / max: {df_viz[year_col].min():.0f} / {df_viz[year_col].max():.0f}
{_rating_line}
Visuals: histogram_release_year.png, countplot_content_type.png,
         scatter_year_vs_duration.png, correlation_heatmap.png
---
"""
print(summary)

## 15. Checklist

- [ ] `visuals/` contains four PNG files.
- [ ] `data/cleaned_n_movies.csv` (or `cleaned_netflix.csv`) opens correctly in Excel or pandas.
- [ ] Update **`reports/final_report.md`** with your name, date, and interpretation.
- [ ] Dependencies: `pip install -r requirements.txt`

To refresh data, replace **`Downloads\netfilx\n_movies.csv`** or set **`DATA_PATH_OVERRIDE`** in section 1.